In [ ]:
# Validate the Model on the Hub Cluster - Federated Learning

- **Current Cluster:** Hub cluster
- **Dataset:** MNIST (Digits 0, 1, 2, 3, 4, 5, 6, 7, 8, 9)

In [ ]:
# check the models
!ls -la /data/models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score
import joblib
import numpy as np
import os
import matplotlib.pyplot as plt

In [ ]:
from flwr_datasets import FederatedDataset
fds = FederatedDataset(dataset="mnist", partitioners={"train": 1})
dataset = fds.load_partition(0, "train").with_format("numpy")
X, y = dataset["image"].reshape((len(dataset), -1)), dataset["label"]
X, y

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=10000, random_state=0)

X_test.shape, X_test, len(y_test), y_test

In [ ]:
def load_model(model_path: str):
    """
    Load the model from disk.

    Args:
        model_path (str): Path to the saved model.

    Returns:
        LogisticRegression: Loaded model.
    """
    print(f"Loading model from {model_path}")
    return joblib.load(model_path)

def evaluate_model(model, X_test, y_test):
    """
    Test the model on the test dataset.

    Args:
        model (LogisticRegression): Trained model.
        X_test (array-like): Test features.
        y_test (array-like): True test labels.

    Returns:
        float: Accuracy of the model on the test dataset.
    """
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

In [ ]:
test_model = load_model('/data/models/2025-01-14-12-53-03.pkl')

In [ ]:
test_accuracy = evaluate_model(test_model, X_test, y_test)
print(f"FL Aggregated Model Accuracy: {test_accuracy:.2f}")

In [ ]:
# fl models
y_pred = test_model.predict(X_test)

plt.figure(figsize=(20,8))

# Plot predicted vs actual labels for some of the test samples
fig, axes = plt.subplots(1, 10, figsize=(15, 3))
start = 30
end = 40
for i in range(start,end):
    ax = axes[i-start]
    ax.imshow(X_test[i].reshape(28, 28), cmap="gray")

    color = "green" if y_pred[i] == y_test[i] else "red"
    ax.set_title(f"FL Pred: {y_pred[i]}", color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()